In [ ]:
raw_data = pd.read_csv(r"c:\Users\Devangna\Downloads\Churn_Modelling.csv")


In [ ]:
data=pd.DataFrame(raw_data)
data.head()


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
print("\n" + "="*60)
print("STEP 1: CHECKING FOR MISSING VALUES")
print("="*60)


print("\nMissing value count per column:")
print(data.isnull().sum())

# ===== STEP 2: REMOVE COLUMNS WITH TOO MANY MISSING VALUES =====
data_null_clm = data.isnull().sum() / data.shape[0] * 100  

print("\nMissing value percentage per column:")
print(data_null_clm)

# Find columns where more than 20% of data is missing
null20_clm_list = data_null_clm[data_null_clm > 20].index

# Remove columns with more than 20% missing data (too much data lost)
data_drop_clm = data.drop(columns=null20_clm_list)

print(f"\nRemoved {len(null20_clm_list)} columns with >20% missing values")
print("Remaining missing values:")
print(data_drop_clm.isnull().sum())

# ===== STEP 3: SEPARATE DATA BY TYPE =====

print("\nData shapes:")
print(f"Original data shape: {data.shape}") 
print(f"After dropping columns with >20% missing: {data_drop_clm.shape}")

data_object = data_drop_clm.select_dtypes(include=['object'])
print(f"Categorical (text) columns shape: {data_object.shape}")

data_numaric = data_drop_clm.select_dtypes(include=['int', 'float'])
print(f"Numerical columns shape: {data_numaric.shape}")

# ===== STEP 4: HANDLE MISSING VALUES IN NUMERICAL COLUMNS =====

print("\n" + "="*60)
print("STEP 2: FILLING MISSING VALUES")
print("="*60)

print("\nFilling numerical columns with MEDIAN:")
numeic_clo = data_numaric.isnull().sum()
print(data_numaric.isnull().sum())

for i in numeic_clo.index:
    data_numaric[i].fillna(data_numaric[i].median(), inplace=True)

print("All missing numerical values filled with median ")
print(data_numaric.isnull().sum())

# ===== STEP 5: HANDLE MISSING VALUES IN CATEGORICAL COLUMNS =====

print("\nFilling categorical columns with MODE (most common value):")
string_clo = data_object.isnull().sum()

for i in string_clo.index:
    data_object[i].fillna(data_object[i].mode()[0], inplace=True)

print(f"All missing categorical values filled")
print(f"Total remaining missing values: {data_object.isnull().sum().sum()}")

print(data_object.head())
# ============================================================
# SECTION 2: ENCODE OBJECT (CATEGORICAL) COLUMNS (SECOND)
# ============================================================

print("\n" + "="*60)
print("STEP 5: ENCODING CATEGORICAL COLUMNS")
print("="*60)
data_encoded = data_object.copy()
# 1. Gender → Binary mapping
if 'Gender' in data_encoded.columns:
    data_encoded['Gender'] = data_encoded['Gender'].map({'Male': 0, 'Female': 1})
    print("   Gender → Binary Encoding (Male=0, Female=1)")

# 2. Geography → One-Hot Encoding
if 'Geography' in data_encoded.columns:
    data_encoded = pd.get_dummies(data_encoded, columns=['Geography'], drop_first=True, dtype='int')
    print("   Geography → One-Hot Encoding")

# 3. Surname → Frequency Encoding
if 'Surname' in data_encoded.columns:
    surname_counts = data_encoded['Surname'].value_counts()
    data_encoded['Surname'] = data_encoded['Surname'].map(surname_counts)
    print("   Surname → Frequency Encoding (count of occurrences)")

print("\nEncoding complete. New shape:", data_encoded.shape)
print(data_encoded.head())

# ============================================================
# SECTION 3: Train_split_test
# ============================================================

final_data = pd.concat([data_numaric, data_encoded], axis=1)
final_data = final_data.loc[:, ~final_data.columns.duplicated()]

print(final_data.shape)

X = final_data.drop(columns=['Exited'])
y = final_data['Exited']

from sklearn.model_selection import train_test_split
x_train, x_test, y_train_target, y_test_target = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(x_train.columns)

# ============================================================
# SECTION 4: Scaling
# ============================================================

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# Fit only on training data
scaled_data_final_X_tarin_arr = scaler.fit_transform(x_train)
scaled_data_final_X_tarin = pd.DataFrame(scaled_data_final_X_tarin_arr, columns=x_train.columns)

# Transform test data using the same scaler
scaled_data_final_X_test_arr = scaler.transform(x_test)
scaled_data_final_X_test = pd.DataFrame(scaled_data_final_X_test_arr, columns=x_test.columns)

print(scaled_data_final_X_tarin.shape)
print(scaled_data_final_X_test.shape)
print(scaled_data_final_X_tarin.describe())
print(scaled_data_final_X_test.describe())



STEP 1: CHECKING FOR MISSING VALUES

Missing value count per column:
RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

Missing value percentage per column:
RowNumber          0.0
CustomerId         0.0
Surname            0.0
CreditScore        0.0
Geography          0.0
Gender             0.0
Age                0.0
Tenure             0.0
Balance            0.0
NumOfProducts      0.0
HasCrCard          0.0
IsActiveMember     0.0
EstimatedSalary    0.0
Exited             0.0
dtype: float64

Removed 0 columns with >20% missing values
Remaining missing values:
RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance   

/tmp/ipykernel_58/2318183452.py:48: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_numaric[i].fillna(data_numaric[i].median(), inplace=True)
/tmp/ipykernel_58/2318183452.py:59: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=

In [ ]:
# ============================================================
# SECTION 5: TensorFlow Neural Network Model
# ============================================================

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Build a simple feedforward neural network
model = Sequential([
    Dense(64, activation='relu', input_shape=(scaled_data_final_X_tarin.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # since target 'Exited' is binary (0/1)
])




/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,073 (12.00 KB)

 Trainable params: 3,073 (12.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])


In [ ]:
# Train the model
history = model.fit(
    scaled_data_final_X_tarin, y_train_target,
    epochs=50,
    batch_size=32,
    validation_split=0.5,
    verbose=1
)

# Evaluate on test data


Epoch 1/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7855 - loss: 0.4940 - val_accuracy: 0.8130 - val_loss: 0.4351
Epoch 2/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8117 - loss: 0.4299 - val_accuracy: 0.8165 - val_loss: 0.4175
Epoch 3/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8263 - loss: 0.4060 - val_accuracy: 0.8305 - val_loss: 0.4043
Epoch 4/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8405 - loss: 0.3854 - val_accuracy: 0.8382 - val_loss: 0.3873
Epoch 5/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8540 - loss: 0.3655 - val_accuracy: 0.8478 - val_loss: 0.3744
Epoch 6/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8577 - loss: 0.3503 - val_accuracy: 0.8503 - val_loss: 0.3665
Epoch 7/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8597 - loss: 0.3420 - val_accuracy: 0.8420 - val_loss: 0.3705
Epoch 8/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8637 - loss: 0.3335 - val_accuracy: 0.

In [ ]:
loss, accuracy = model.evaluate(scaled_data_final_X_test, y_test_target, verbose=0)
print(f"Test Accuracy: {accuracy:.4f}")



Test Accuracy: 0.8355


In [ ]:
# Predictions
y_pred = model.predict(scaled_data_final_X_test)
print("Sample predictions:", y_pred[:10].flatten())

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Sample predictions: [0.00174987 0.00060906 0.12761895 0.08909249 0.44791454 0.00132587
 0.40181178 0.05189588 0.02864141 0.33334127]


In [ ]:
y2=np.where(y_pred>0.5,1,0)

In [ ]:
y2


array([[0],
       [0],
       [0],
       ...,
       [1],
       [0],
       [0]], shape=(2000, 1))

0.8355